[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/magrilu/cv-dojo/blob/main/notebooks/camera/resection.ipynb)

# Camera resection

A camera maps 3D points to image points. Camera resection asks the inverse question: if the 3D scene is known and some of its points have been identified in a photograph, can we recover the camera that produced it?

This is **camera resection**, or *exterior orientation* in photogrammetry. It is the counterpart of [intersection](../reconstruction/triangulation.ipynb): intersection recovers a 3D point from known cameras; resection recovers a camera from known 3D points.

Let's recall the notation introduced for [perspective cameras](../camera/camera.ipynb). A point in space $\mathbb{P}^3$ can be written in homogeneous coordinates as $\mathbf{X}=(X,Y,Z,1)^\top,$ and its image in $\mathbb{P}^2$, as $\mathbf x=(u,v,1)^\top.$

A projective camera is represented by a $3\times4$ matrix $\mathsf P$ satisfying

$$
\mathbf x \sim \mathsf P\mathbf X,
$$

where $\sim$ means *equal up to a non-zero scale*. Given $n$ correspondences

$$
\mathbf X_i \leftrightarrow \mathbf x_i,
$$

our task is to estimate $\mathsf P$.

We will first solve the problem with the **direct linear transform (DLT)** on
synthetic data, where the true camera is known, and then apply the same method
to photographs of the origami house. Along the way we will see why coordinate
normalization matters, how the linear estimate can be refined using
reprojection error, and which 3D configurations fail to determine a unique
camera.



In [ ]:
#| echo: false
import sys, subprocess
from pathlib import Path

if "google.colab" in sys.modules and not Path("cv-dojo").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/magrilu/cv-dojo.git"], check=True)
    import os
    os.chdir("cv-dojo")

for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "src" / "cvdojo").exists():
        sys.path.insert(0, str(parent / "src"))
        ROOT = parent
        break

import numpy as np
import matplotlib.pyplot as plt

from cvdojo.house import load_model, load_annotation, load_image
from cvdojo.plotting import ACCENT

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (8, 5), "axes.grid": False})

BLUE, GREY, RED, DARK = "#3288BD", "0.45", "#C0392B", "0.20"

model = load_model()
V3 = {k: np.array(v, float).reshape(3, 1) for k, v in model["vertices"].items()}
IDS, EDGES = list(V3), model["edges"]
index = {k: i for i, k in enumerate(IDS)}

# Points are columns: X_ALL is 3 x 10, one vertex per column.
X_ALL = np.hstack([V3[k] for k in IDS])

W_PIX, H_PIX = 1536, 2048          # the format of the real photographs

In [ ]:
#| echo: false


def reprojection_error(P, X, x):
    """Distance in pixels between the projected points and the measured ones."""
    return np.linalg.norm(project_points(X, P) - np.asarray(x, float), axis=0)


def rms(e):
    return float(np.sqrt(np.mean(np.asarray(e) ** 2)))


def wireframe_2d(ax, x, color=BLUE, lw=2.0, alpha=1.0, dots=False, zorder=4):
    """Draw the house edges from a 2 x N array of image points."""
    for a, b in EDGES:
        q = x[:, [index[a], index[b]]]
        ax.plot(q[0], q[1], lw=lw, color=color, alpha=alpha, zorder=zorder)
    if dots:
        ax.scatter(x[0], x[1], s=20, color=color, alpha=alpha, zorder=zorder)


def wireframe_3d(ax, X, color=BLUE, lw=1.8, dots=False, alpha=1.0):
    """Draw the house edges from a 3 x N array of world points."""
    for a, b in EDGES:
        q = X[:, [index[a], index[b]]]
        ax.plot(q[0], q[1], q[2], lw=lw, color=color, alpha=alpha)
    if dots:
        ax.scatter(X[0], X[1], X[2], s=18, color=color, alpha=alpha)


def image_frame(ax, color=GREY):
    ax.plot([0, W_PIX, W_PIX, 0, 0], [0, 0, H_PIX, H_PIX, 0], lw=1.0, ls="--",
            color=color)
    ax.set_xlim(-40, W_PIX + 40); ax.set_ylim(H_PIX + 40, -40)
    ax.set_aspect("equal"); ax.axis("off")


def set_axes_equal(ax):
    lims = np.array([ax.get_xlim3d(), ax.get_ylim3d(), ax.get_zlim3d()])
    c, r = lims.mean(axis=1), 0.5 * np.ptp(lims, axis=1).max()
    ax.set_xlim3d(c[0]-r, c[0]+r)
    ax.set_ylim3d(c[1]-r, c[1]+r)
    ax.set_zlim3d(c[2]-r, c[2]+r)


def draw_camera(ax, P, size=3.0, color=ACCENT, lw=1.6, label=None):
    """The camera as a little pyramid: its centre, and the cone of its frame."""
    K, R, t = decompose_camera(P)
    C = -R.T @ t                                            # 3 x 1

    corners = np.array([[0, W_PIX, W_PIX, 0],
                        [0, 0, H_PIX, H_PIX]], float)       # 2 x 4

    rays = np.vstack((
        (corners - K[:2, 2:3]) / np.diag(K)[:2, None],
        np.ones((1, 4)),
    ))
    rays = R.T @ (size * rays / np.linalg.norm(rays, axis=0)) + C

    for j in range(rays.shape[1]):
        ax.plot(*np.hstack([C, rays[:, [j]]]), color=color, lw=lw * 0.7)

    loop = np.hstack([rays, rays[:, [0]]])
    ax.plot(loop[0], loop[1], loop[2], color=color, lw=lw)
    ax.scatter(*C[:, 0], s=45, color=color, edgecolors="white", linewidths=1.0,
               zorder=9)

    if label:
        ax.text(*(C[:, 0] + np.array([0.0, 0.0, 0.35 * size])), label, color=color,
                fontsize=10, ha="center")

    return C

## A synthetic image

Before turning to a real photograph, let's start with a controlled synthetic case.

We already know the 3D coordinates of the ideal origami house. Choose a camera
$\mathsf P_{\mathrm{syn}}$ and use it to project the house vertices into an image, $\mathbf x_i \sim \mathsf P_{\mathrm{syn}} \mathbf X_i.$

We then keep only the resulting 3D--2D correspondences
$\mathbf X_i \leftrightarrow \mathbf x_i$.

The resection problem is now simple to state: recover
$\mathsf P_{\mathrm{syn}}$ from these correspondences alone.

To produce our synthetic image, we must choose three things:

1. where the camera is, given by its centre $\mathbf C_{\mathrm{syn}}$;
2. where it looks, encoded by its orientation $\mathsf R_{\mathrm{syn}}$;
3. how it forms the image, encoded by its intrinsic matrix $\mathsf K_{\mathrm{syn}}$.

Together they determine the camera matrix

$$
\mathsf P_{\mathrm{syn}}
=
\mathsf K_{\mathrm{syn}}
\begin{bmatrix}
\mathsf R_{\mathrm{syn}} &
-\mathsf R_{\mathrm{syn}}\mathbf C_{\mathrm{syn}}
\end{bmatrix}.
$$

We can then project the vertices of the origami house via $\mathsf P_{\mathrm{syn}}$,

$$
\mathbf x_i \sim \mathsf P_{\mathrm{syn}}\mathbf X_i,
$$

to obtain the synthetic 3D--2D correspondences that will serve as input to the resection problem.

In [ ]:
# Where is the camera?
# Place the synthetic camera away from the house.
# Its position is expressed in the same 3D coordinate system as the house.
house_centre = X_ALL.mean(axis=1, keepdims=True)
C_syn = house_centre + np.array([[9.0], [-16.0], [11.0]])


# Where does it look?
def look_at(centre, target, up=np.array([[0.0], [0.0], [1.0]])):
    """Build a rotation for a camera located at `centre` and looking at `target`."""

    # Camera z-axis: viewing direction.
    z = target - centre
    z = z / np.linalg.norm(z)

    # Camera x-axis: perpendicular to the viewing direction and world vertical.
    x = np.cross(z[:, 0], up[:, 0])[:, None]
    x = x / np.linalg.norm(x)

    # Camera y-axis completes the orthonormal frame.
    # With this convention it points downwards in the image.
    y = np.cross(z[:, 0], x[:, 0])[:, None]

    # The rows of R are the camera axes expressed in world coordinates.
    return np.vstack((x.T, y.T, z.T))


R_syn = look_at(C_syn, house_centre)


# What are the intrinsics?
# Choose simple intrinsics: square pixels, no skew,
# and the principal point at the centre of the image.
def intrinsic_matrix(fx, fy=None, cx=0.0, cy=0.0, skew=0.0):
    if fy is None:
        fy = fx
    return np.array([
        [fx, skew, cx],
        [0.0, fy, cy],
        [0.0, 0.0, 1.0],
    ])


K_syn = intrinsic_matrix(2900.0, cx=W_PIX / 2, cy=H_PIX / 2)

# We can assemble the camera projection matrix:
#
#                    P = K [ R | -RC ].
#
# C_syn is the camera centre in world coordinates;
# R_syn expresses world points in the camera coordinate frame.
P_syn = K_syn @ np.hstack((R_syn, -R_syn @ C_syn))


def project_points(X, P):
    """Project 3D points, given as columns, into ordinary image coordinates."""
    Xh = np.vstack((X, np.ones((1, X.shape[1]))))     # homogeneous, 4 x n
    xh = P @ Xh                                       # homogeneous image, 3 x n
    return xh[:2] / xh[2:3]                           # divide through, 2 x n


# These image points are the synthetic measurements that will
# be the input of the resection algorithm.
x_syn = project_points(X_ALL, P_syn)

print("the synthetic camera stands at", np.round(C_syn.ravel(), 2), "cm")
print(
    "and its picture of the house spans",
    np.round(np.ptp(x_syn, axis=1)).astype(int),
    f"px inside a {W_PIX} x {H_PIX} frame",
)

In [ ]:
#| echo: false
#| label: fig-synthetic-view
#| fig-cap: >-
#|   The synthetic photograph. Ten vertices, projected by a camera whose position,
#|   orientation and focal length we chose ourselves; this and the metric model are the
#|   only inputs the estimator gets.
fig, ax = plt.subplots(figsize=(4.4, 5.6), layout="constrained")
wireframe_2d(ax, x_syn, dots=True)
image_frame(ax)
plt.show()

## From one correspondence to two equations

A single 3D--2D correspondence satisfies

$$
\mathbf x_i \sim \mathsf P \mathbf X_i.
$$

More explicitly, there exists a
non-zero scalar $\lambda_i \in \mathbb{R}-\{0\}$ such that

$$
\mathsf P \mathbf X_i = \lambda_i \mathbf x_i.
$$

Instead of dealing with such scale factors $\lambda_i$ for every correspondence we can eliminate them. Two homogeneous vectors represent the same image
point precisely when they are parallel, hence

$$
\mathbf x_i \times \mathsf P\mathbf X_i = \mathbf 0,
$$

or, using the skew-symmetric matrix of $\mathbf x_i$,

$$
[\mathbf x_i]_\times \mathsf P\mathbf X_i = \mathbf 0.
$$

The unknown scale $\lambda_i$ has disappeared.

Now write the rows of $\mathsf P$ as
$\mathbf p^{1\top},\mathbf p^{2\top},\mathbf p^{3\top}$ and stack them into

$$
\mathbf p
=
\begin{bmatrix}
\mathbf p^1\\
\mathbf p^2\\
\mathbf p^3
\end{bmatrix}
=
\operatorname{vec}(\mathsf P^\top).
$$

Since

$$
\mathsf P\mathbf X_i
=
(\mathsf I_3\otimes\mathbf X_i^\top)\mathbf p,
$$

one correspondence gives the homogeneous linear system

$$
\underbrace{
\left(
[\mathbf x_i]_\times\otimes\mathbf X_i^\top
\right)
}_{\mathsf A_i}
\mathbf p
=
\mathbf 0.
$$

This is a $3\times12$ system, but it contains only **two independent
equations**. Indeed, for every non-zero image point,

$$
\operatorname{rank}[\mathbf x_i]_\times=2,
$$

and therefore

$$
\operatorname{rank}\mathsf A_i=2.
$$

For $\mathbf x_i=(u_i,v_i,1)^\top$, the three rows are

$$
\begin{bmatrix}
\mathbf 0^\top & -\mathbf X_i^\top & v_i\mathbf X_i^\top\\
\mathbf X_i^\top & \mathbf 0^\top & -u_i\mathbf X_i^\top\\
-v_i\mathbf X_i^\top & u_i\mathbf X_i^\top & \mathbf 0^\top
\end{bmatrix}
\mathbf p
=
\mathbf 0.
$$

These three equations are not independent. If we call the rows
$\mathbf R_1,\mathbf R_2,\mathbf R_3$, then

$$
\mathbf R_3
=
-u_i\,\mathbf R_1
-v_i\,\mathbf R_2.
$$


So a single 3D--2D correspondence contributes only two independent linear
constraints.

We can therefore keep, for example, an equivalent pair of independent equations:

$$
\begin{bmatrix}
\mathbf X_i^\top & \mathbf 0^\top & -u_i\mathbf X_i^\top\\
\mathbf 0^\top & \mathbf X_i^\top & -v_i\mathbf X_i^\top
\end{bmatrix}
\mathbf p
=
\mathbf 0.
$$

For each 3D--2D correspondence we now have two linear equations in the twelve
entries of the camera matrix. The following function simply writes those two
rows for every correspondence and stacks them into a single matrix $\mathsf A$.


In [ ]:
def camera_design_matrix(X, x):
    """Build the 2n x 12 design matrix A for camera resection.

    Points are columns; the rows of A are equations, two per correspondence.
    """
    n = X.shape[1]
    Q = np.vstack((np.asarray(X, float), np.ones((1, n))))
    x = np.asarray(x, float)

    A = np.zeros((2 * n, 12))

    for i in range(n):
        Xi = Q[:, [i]]          # 4 x 1
        u = x[0, i]
        v = x[1, i]

        A[2*i] = np.hstack((
            Xi.T,
            np.zeros((1, 4)),
            -u * Xi.T,
        ))

        A[2*i + 1] = np.hstack((
            np.zeros((1, 4)),
            Xi.T,
            -v * Xi.T,
        ))

    return A

## The Direct Linear Transform

Stacking all correspondences gives

$$
\mathsf A\mathbf p=\mathbf 0,
\qquad
\mathsf A\in\mathbb R^{2n\times12}.
$$

At this point the geometry has been converted into a linear algebra problem.

There is one important detail: this is a **homogeneous** system. The zero vector
would of course satisfy it, but it does not represent a valid camera that must have rank 3. What we want is
a non-zero vector solution $\mathbf p$, and we only need it up to scale because

$$
\mathsf P
\quad\text{and}\quad
\alpha\mathsf P,
\qquad \alpha\neq0,
$$

represent the same projective camera.

With exact correspondences in general position, $\mathsf A$ has rank $11$.
Its null space is therefore one-dimensional, and any non-zero vector spanning
that null space gives the camera matrix up to scale.

With noisy measurements there is generally no exact null vector. We therefore
look for the solution that comes closest:

$$
\min_{\|\mathbf p\|=1}
\|\mathsf A\mathbf p\|^2.
$$

The unit-norm constraint simply fixes the scale and rules
out the trivial solution $\mathbf p=\mathbf 0$. The solution is the right
singular vector of $\mathsf A$ associated with its smallest singular value, and can be retrieved via SVD.

This is the **direct linear transform**, or **DLT**.

In [ ]:
def dlt_camera(X, x):
    """Estimate the camera matrix from n >= 6 3D--2D correspondences."""
    A = camera_design_matrix(X, x)

    # The solution of min ||A p|| subject to ||p|| = 1
    # is the right singular vector associated with the smallest singular value.
    _, _, Vt = np.linalg.svd(A)
    p = Vt[-1]

    P = p.reshape(3, 4)

    # Choose one representative of the projective camera P ~ alpha P.
    return P / np.linalg.norm(P)


A_syn = camera_design_matrix(X_ALL, x_syn)
P_hat = dlt_camera(X_ALL, x_syn)

print(
    f"A has shape {A_syn.shape} "
    f"and rank {np.linalg.matrix_rank(A_syn)}."
)

print(
    f"largest reprojection error: "
    f"{reprojection_error(P_hat, X_ALL, x_syn).max():.2e} px"
)

For our exact synthetic correspondences, $\mathsf A$ has rank $11$, so its
null space is one-dimensional, as expected. The recovered matrix
reprojects the points essentially to machine precision.

> **Where does the name come from?**  
> The DLT was introduced by Y. I. Abdel-Aziz and H. M. Karara in 1971,
> in close-range photogrammetry. Their problem already looks remarkably
> familiar: given the known 3D coordinates of control points and their measured
> 2D positions in a photograph, estimate the camera that relates them.
>
> **Direct** meant estimating this 3D-to-2D mapping in one step, rather than
> first recovering several intermediate photogrammetric calibration and
> orientation parameters. **Linear** meant that the unknown coefficients could
> be obtained from linear equations. In modern computer-vision notation, these
> coefficients are the entries of the projective camera matrix $\mathsf P$,
> determined up to scale.

## How many correspondences?

We can now count the number of 2D-3D correspondences needed to get the perspective camera matrix.

A $3\times4$ camera matrix has twelve entries, but its global scale is
irrelevant:

$$
12-1=11
\qquad\text{degrees of freedom.}
$$

Each 3D--2D correspondence contributes two independent linear constraints, so

$$
2n\geq11.
$$

Thus at least

$$
n=6
$$

complete correspondences are required.


All of this assumes that the correspondences are in **general position**.
In particular, points lying on a single plane cannot determine a general
$3\times4$ camera matrix.

There is no additional rank constraint to enforce here. A non-degenerate
projective camera is simply a $3\times4$ matrix of rank three, and a generic
DLT estimate already has rank three.

This is different from, for example, the [fundamental matrix](../two-views/eight-point.ipynb), whose rank-two
constraint must be explicitly restored after a linear estimate.

At this stage we are making no Euclidean assumptions about the camera:
we do not require a calibration matrix $\mathsf K$, an orthogonal rotation
$\mathsf R$, or a decomposition of $\mathsf P$. We have recovered a
**projective camera**.

There is one geometric object that can be read from any
projective camera: its **camera centre**.

In [ ]:
def camera_centre(P):
    """Return the finite camera centre as a 3 x 1 column."""
    _, _, Vt = np.linalg.svd(np.asarray(P, float))

    # The last right singular vector spans null(P).
    C = Vt[-1]

    # Convert the homogeneous point C ~ (X, Y, Z, W) to Euclidean coordinates.
    return (C[:3] / C[3]).reshape(3, 1)


C_hat = camera_centre(P_hat)

print("true camera centre:     ", np.round(C_syn.ravel(), 6), "cm")
print("recovered camera centre:", np.round(C_hat.ravel(), 6), "cm")
print(
    f"error on the camera centre: "
    f"{np.linalg.norm(C_hat - C_syn):.2e} cm"
)

This closes the synthetic experiment. From the 3D-2D correspondences alone, the DLT recovered
a projective camera whose null space brings us back to the original viewpoint.


## From a projective matrix to a camera

The DLT treated $\mathsf P$ simply as a projective $3\times4$ matrix. It did not exploit during the estimation of $\mathsf P$ any structure on its entries involving the calibration matrix, or the rotation.

But the coordinates used in our synthetic experiment were not arbitrary
projective coordinates: the origami house was described in a Euclidean 3D reference frame, and its image points were expressed in pixel coordinates. We can therefore ask for the usual Euclidean interpretation of the recovered camera.

Write

$$
\mathsf P
=
\begin{bmatrix}
\mathsf M & \mathbf p_4
\end{bmatrix},
$$

where $\mathsf M$ is the left $3\times3$ block. For a finite perspective camera,

$$
\mathsf P
=
\mathsf K
\begin{bmatrix}
\mathsf R & \mathbf t
\end{bmatrix},
$$

so in particular

$$
\mathsf M = \mathsf K\mathsf R.
$$

Here $\mathsf K$ is upper triangular and contains the intrinsic parameters,
while $\mathsf R$ is a rotation. This is exactly the structure produced by an
**RQ decomposition** of $\mathsf M$:

$$
\mathsf M = \mathsf K\mathsf R.
$$

The factorization is not quite unique: signs can be moved between
$\mathsf K$ and $\mathsf R$, and the whole camera is still defined only up to
scale. We therefore choose the usual convention of a positive diagonal for
$\mathsf K$, enforce $\det\mathsf R=+1$, and normalize $\mathsf K_{33}=1$.

Once $\mathsf K$ is known, the last column of $\mathsf P$ gives the translation,

$$
\mathbf t
=
\mathsf K^{-1}\mathbf p_4.
$$

In [ ]:
def rq3(A):
    """Factor A = K R, with K upper triangular and R orthogonal."""
    Q, R = np.linalg.qr(np.flipud(np.asarray(A, float)).T)
    K = np.fliplr(np.flipud(R.T))
    R = np.flipud(Q.T)
    return K, R


def decompose_camera(P):
    """Decompose a finite camera as P ~ K [R | t], with t a 3 x 1 column."""
    P = np.asarray(P, float).copy()

    # Choose the global sign so that the left 3x3 block
    # can have a positive-determinant calibration matrix.
    if np.linalg.det(P[:, :3]) < 0:
        P = -P

    K, R = rq3(P[:, :3])

    # Move signs from K into R so that diag(K) is positive.
    D = np.diag(np.where(np.diag(K) < 0, -1.0, 1.0))
    K = K @ D
    R = D @ R

    # Fix the remaining projective scale by setting K[2,2] = 1.
    scale = K[2, 2]
    K = K / scale
    P = P / scale

    t = np.linalg.solve(K, P[:, 3:4])

    return K, R, t


K_rec, R_rec, t_rec = decompose_camera(P_hat)

print("K recovered:\n", K_rec)

print(
    f"\nlargest deviation from the synthetic K: "
    f"{np.abs(K_rec - K_syn).max():.2e}"
)

print(
    f"largest deviation from the synthetic R: "
    f"{np.abs(R_rec - R_syn).max():.2e}"
)

## Normalising the coordinates

Before adding noise, it is worth looking at the linear system itself. The
non-zero entries of $\mathsf A$ do not all have the same numerical scale.

In [ ]:
A_syn = camera_design_matrix(X_ALL, x_syn)

for name, block in [
    (r"p1", A_syn[:, 0:4]),
    (r"p2", A_syn[:, 4:8]),
    (r"p3", A_syn[:, 8:12]),
]:
    values = np.abs(block)
    values = values[values > 0]
    print(f"{name}: {values.min():.2g}  to  {values.max():.2g}")

The design matrix depends on the coordinate systems in which the points are
expressed. In our example, the 3D coordinates of the house are a few units
across, while the image coordinates are measured in thousands of pixels.
Consequently, terms such as $u_i\mathbf X_i$ and $v_i\mathbf X_i$ can be much
larger than the coordinates $\mathbf X_i$ that appear in the other columns of
$\mathsf A$.

This does not change the projective geometry, but it can make the linear
system unnecessarily sensitive to numerical errors.

A standard remedy is to change coordinates before building the system. Let

$$
\widehat{\mathbf X}_i = \mathsf U\,\mathbf X_i
$$

be a similarity transformation that recentres and rescales the 3D points, and

$$
\widehat{\mathbf x}_i = \mathsf T\,\mathbf x_i
$$

the corresponding normalization of the image points.

The camera relating the normalized coordinates is not $\mathsf P$. Starting
from

$$
\mathbf x_i \sim \mathsf P\mathbf X_i,
$$

we obtain

$$
\widehat{\mathbf x}_i
\sim
\mathsf T\,\mathsf P\,\mathsf U^{-1}
\widehat{\mathbf X}_i.
$$

Therefore

$$
\widehat{\mathsf P}
=
\mathsf T\,\mathsf P\,\mathsf U^{-1}.
$$

We estimate $\widehat{\mathsf P}$ from the normalized correspondences and then
return to the original coordinates with

$$
\mathsf P
=
\mathsf T^{-1}\widehat{\mathsf P}\,\mathsf U.
$$

The two normalizations act in different spaces:
$\mathsf U$ is a $4\times4$ transformation of the 3D world coordinates,
whereas $\mathsf T$ is a $3\times3$ transformation of the image coordinates.

In [ ]:
#| echo: false
#| label: fig-normalisation
#| fig-cap: >-
#|   Normalisation as a change of coordinates. The camera is estimated along the bottom
#|   edge, where the numbers are well behaved, and carried back to pixels and centimetres
#|   along the sides.
fig, ax = plt.subplots(figsize=(7.0, 4.0), layout="constrained")
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis("off")
pos = {"p3": (1.5, 4.6), "p2": (8.5, 4.6), "n3": (1.5, 1.2), "n2": (8.5, 1.2)}
lab = {"p3": r"$\mathbb{P}^3$", "p2": r"$\mathbb{P}^2$",
       "n3": r"$\widehat{\mathbb{P}}^3$", "n2": r"$\widehat{\mathbb{P}}^2$"}
for k, (xx, yy) in pos.items():
    ax.text(xx, yy, lab[k], fontsize=17, ha="center", va="center")
for a, b, t, off in [("p3", "p2", r"$\mathsf{P}$", 0.35),
                     ("n3", "n2", r"$\widehat{\mathsf{P}}$", -0.55),
                     ("p3", "n3", r"$\mathsf{U}$", 0.0),
                     ("p2", "n2", r"$\mathsf{T}$", 0.0)]:
    (x0, y0), (x1, y1) = pos[a], pos[b]
    dx, dy = x1 - x0, y1 - y0
    n = np.hypot(dx, dy); ux, uy = dx / n, dy / n
    ax.annotate("", xy=(x1 - 0.7*ux, y1 - 0.45*uy), xytext=(x0 + 0.7*ux, y0 + 0.45*uy),
                arrowprops=dict(arrowstyle="-|>", color=GREY, lw=1.3))
    ax.text((x0+x1)/2 + (0 if dx else 0.45), (y0+y1)/2 + off, t,
            fontsize=14, color=ACCENT, ha="center", va="center")
ax.set_title(r"$\widehat{\mathsf{P}} = \mathsf{T}\,\mathsf{P}\,\mathsf{U}^{-1}$",
             fontsize=12)
plt.show()

We use the standard isotropic normalization: translate each point set so that
its centroid is at the origin, then rescale it so that the mean distance from
the origin is $\sqrt{2}$ in the image and $\sqrt{3}$ in space.

In [ ]:
def normalise_2d(x):
    """Centroid at the origin, mean distance equal to sqrt(2)."""
    x = np.asarray(x, float)

    c = x.mean(axis=1, keepdims=True)
    s = np.sqrt(2) / np.linalg.norm(x - c, axis=0).mean()

    T = np.array([
        [s, 0, -s * c[0, 0]],
        [0, s, -s * c[1, 0]],
        [0, 0, 1.0],
    ])

    return (x - c) * s, T


def normalise_3d(X):
    """Centroid at the origin, mean distance equal to sqrt(3)."""
    X = np.asarray(X, float)

    c = X.mean(axis=1, keepdims=True)
    s = np.sqrt(3) / np.linalg.norm(X - c, axis=0).mean()

    U = np.array([
        [s, 0, 0, -s * c[0, 0]],
        [0, s, 0, -s * c[1, 0]],
        [0, 0, s, -s * c[2, 0]],
        [0, 0, 0, 1.0],
    ])

    return (X - c) * s, U


def resection(X, x):
    """Normalised DLT, carried back to the original coordinates."""
    Xn, U = normalise_3d(X)
    xn, T = normalise_2d(x)

    Pn = dlt_camera(Xn, xn)
    P = np.linalg.inv(T) @ Pn @ U

    return P / np.linalg.norm(P)

The geometry has not changed: we have only changed coordinates before solving
the linear system. What does change is its numerical scale.

To see this, we can move the origin of the world farther and farther away from
the house. This describes exactly the same scene in increasingly inconvenient
coordinates.

In [ ]:
def conditioning(A):
    """Spread of the singular values that are not forced to vanish."""
    s = np.linalg.svd(A, compute_uv=False)
    return s[0] / s[-2]


print(
    f"{'world offset':>14s}"
    f"{'raw system':>14s}"
    f"{'normalised':>14s}"
    f"{'error, raw':>16s}"
    f"{'error, normalised':>20s}"
)

for offset in [0.0, 1e2, 1e4, 1e6]:
    Xs = X_ALL + offset
    C_expected = C_syn + offset

    Xn, _ = normalise_3d(Xs)
    xn, _ = normalise_2d(x_syn)

    e_raw = np.linalg.norm(camera_centre(dlt_camera(Xs, x_syn)) - C_expected)
    e_norm = np.linalg.norm(camera_centre(resection(Xs, x_syn)) - C_expected)

    print(
        f"{offset:14.0e}"
        f"{conditioning(camera_design_matrix(Xs, x_syn)):14.1e}"
        f"{conditioning(camera_design_matrix(Xn, xn)):14.1e}"
        f"{e_raw:16.2e}"
        f"{e_norm:20.2e}"
    )

In [ ]:
#| echo: false
#| label: fig-design-scales
#| fig-cap: >-
#|   Orders of magnitude of the non-zero entries of the design matrix, before and after
#|   normalisation. The block structure is identical; only the numerical scale changes.

def log_magnitudes(A):
    """log10 of the non-zero magnitudes, with the structural zeros masked."""
    mag = np.abs(A)
    return np.ma.masked_where(
        mag == 0,
        np.log10(np.maximum(mag, np.finfo(float).tiny)),
    )


Xn, _ = normalise_3d(X_ALL)
xn, _ = normalise_2d(x_syn)

L_raw = log_magnitudes(camera_design_matrix(X_ALL, x_syn))
L_norm = log_magnitudes(camera_design_matrix(Xn, xn))

values = np.r_[L_raw.compressed(), L_norm.compressed()]
vmin, vmax = values.min(), values.max()

fig, axes = plt.subplots(1, 2, figsize=(9.0, 4.2), layout="constrained")

for ax, L, title in zip(
    axes, [L_raw, L_norm], ["original coordinates", "normalised coordinates"]
):
    im = ax.imshow(L, aspect="auto", interpolation="nearest", vmin=vmin, vmax=vmax)
    ax.set_xticks([1.5, 5.5, 9.5])
    ax.set_xticklabels([r"$\mathbf{p}^1$", r"$\mathbf{p}^2$", r"$\mathbf{p}^3$"])
    ax.set_xlabel("camera parameters")
    ax.set_title(title, fontsize=10)

axes[0].set_ylabel("equations")
axes[1].set_yticklabels([])

cbar = fig.colorbar(im, ax=axes, shrink=0.9)
cbar.set_label(r"$\log_{10}|A_{ij}|$")

plt.show()

The block structure is unchanged, as it should be: normalization does not
change the DLT equations, it changes just their numerical scale.

In the original coordinates, terms involving image coordinates, such as
$u_i\mathbf X_i$ and $v_i\mathbf X_i$, can be much larger than the other
entries of $\mathsf A$. After normalization, the non-zero coefficients are
brought to more comparable scales.

The heatmap gives a visual indication of this balancing. The singular values
make the effect quantitative: as the world coordinates become poorly scaled,
the conditioning of the unnormalised system deteriorates, while the normalised
system remains much more stable.

In this synthetic example the recovered camera itself changes very little.
The main role of normalization is therefore numerical: it reduces the
dependence of the linear estimate on the particular coordinate systems in
which the problem is expressed.

## Are six points always enough?

Six correspondences are sufficient to determine a projective camera in general
position. That is a statement about uniqueness with exact data, not about
accuracy in the presence of noise.

We can test the difference while the true camera is still known. Add a few
pixels of noise to the image measurements, estimate the camera from subsets of
increasing size, and ask two questions:

- how accurately is the camera centre recovered?
- how well does the estimated camera describe the projection of the noise-free image points?

If additional correspondences provide useful redundancy, both errors should
decrease as more points are used.

In [ ]:
rng = np.random.default_rng(7)


def degeneracy(sel):
    """Smallest singular value that should not vanish, relative to the largest."""
    Xn, _ = normalise_3d(X_ALL[:, sel])
    xn, _ = normalise_2d(x_syn[:, sel])
    s = np.linalg.svd(camera_design_matrix(Xn, xn), compute_uv=False)
    return s[-2] / s[0]


USABLE = 1e-8          # below this value the configuration is critical, not merely awkward

noise_sigma = 3.0
n_trials = 300
n_points = X_ALL.shape[1]

print(
    f"{'points':>8s}"
    f"{'equations':>12s}"
    f"{'RMS on all vertices':>22s}"
    f"{'centre error':>18s}"
)

for k in range(6, n_points + 1):

    image_errors = []
    centre_errors = []

    while len(centre_errors) < n_trials:

        sel = rng.choice(n_points, k, replace=False)

        if degeneracy(sel) < USABLE:
            continue

        x_noisy = x_syn + rng.normal(0.0, noise_sigma, x_syn.shape)

        P = resection(X_ALL[:, sel], x_noisy[:, sel])

        image_errors.append(rms(reprojection_error(P, X_ALL, x_syn)))
        centre_errors.append(np.linalg.norm(camera_centre(P) - C_syn))

    print(
        f"{k:8d}"
        f"{2*k:12d}"
        f"{np.median(image_errors):19.2f} px"
        f"{np.median(centre_errors):15.2f} cm"
    )

The distinction between *minimal* and *well constrained* is now visible.
Six points determine the camera, but leave almost no redundancy with which to
average measurement noise. Adding correspondences does not change the model:
it only gives the same eleven camera degrees of freedom more evidence.

This is why practical resection is normally solved from more than the minimum
number of correspondences whenever they are available.

## On real images

Let us now try the DLT on real photographs.

For each image, the house corners have been annotated by hand, giving us the
2D image measurements $\mathbf x_i$. Their corresponding 3D coordinates
$\mathbf X_i$ come from the ideal metric model of the origami house.

Unlike the synthetic experiment, neither side of these correspondences is exact.

The image points are affected by annotation noise: a corner can only be located
to within a few pixels. But the 3D model is approximate as well. It describes
the nominal geometry of the folded house, not a careful metrological
measurement of the particular paper model in the photograph. Small folding
errors, bends, and deformations therefore appear as errors in the 3D points.

We should therefore not expect an exact camera. The question is instead how
well a single projective matrix can explain these imperfect 3D--2D
correspondences.

In [ ]:
matches = [
    m for m in load_annotation("two_view_matches")["matches"]
    if m["id"] in V3
]

ids = [m["id"] for m in matches]

X  = np.hstack([V3[i] for i in ids])
x1 = np.array([m["x"]  for m in matches], float).T
x2 = np.array([m["xp"] for m in matches], float).T

I1 = load_image("IMG_4331.jpeg")
I2 = load_image("IMG_4337.jpeg")

views = [
    ("IMG_4331", I1, x1),
    ("IMG_4337", I2, x2),
]

print(f"using {len(ids)} annotated vertices:", ", ".join(ids))


P1 = resection(X, x1)
P2 = resection(X, x2)

for (name, _, x), P in zip(views, [P1, P2]):

    e = reprojection_error(P, X, x)
    C = camera_centre(P)

    print(f"\n{name}")
    print(f"  RMS reprojection error: {rms(e):.2f} px")
    print(f"  largest error:          {e.max():.2f} px")
    print(f"  estimated camera centre: {np.round(C.ravel(), 2)} cm")

In [ ]:
#| echo: false
#| column: page
#| label: fig-real-resection
#| fig-cap: >-
#|   The recovered cameras reproject the nominal 3D house onto the two
#|   photographs. White circles mark the image points used for resection.

fig, axes = plt.subplots(1, 2, figsize=(13.5, 7.5), layout="constrained")

for ax, (name, image, x), P in zip(axes, views, [P1, P2]):

    ax.imshow(image)

    # Reproject the complete nominal house.
    x_hat = project_points(X_ALL, P)
    wireframe_2d(ax, x_hat, color=ACCENT, lw=2.0)

    # The six annotated points used for resection.
    ax.scatter(
        x[0], x[1],
        s=70,
        facecolors="none",
        edgecolors="white",
        linewidths=1.5,
        zorder=10,
    )

    e = reprojection_error(P, X, x)
    ax.set_title(f"{name}: RMS {rms(e):.1f} px")
    ax.axis("off")

plt.show()

The reprojected wireframes follow the house reasonably well, with residuals of a few
pixels. To see what the two recovered cameras mean geometrically, let us place them in
the 3D scene at their estimated centres and with their recovered orientations.

In [ ]:
#| echo: false
#| column: page
#| label: fig-two-cameras-3d
#| fig-cap: >-
#|   The two recovered viewpoints. **Left and centre:** each photograph on its own.
#|   **Right:** both together, which is the pair of positions the two-view notebooks call
#|   a baseline — recovered here one camera at a time, from the model alone.
fig = plt.figure(figsize=(13.5, 4.8), layout="constrained")
panels = [("IMG_4331", [P1], [ACCENT]), ("IMG_4337", [P2], [BLUE]),
          ("both", [P1, P2], [ACCENT, BLUE])]
for j, (ttl, Ps, cols) in enumerate(panels):
    ax = fig.add_subplot(1, 3, j + 1, projection="3d")
    wireframe_3d(ax, X_ALL, color=GREY, lw=1.4)
    for P, c, lab in zip(Ps, cols, ["4331", "4337"] if len(Ps) == 2 else [None]):
        draw_camera(ax, P, size=6.0, color=c, label=lab)
    ax.set_title(ttl, fontsize=10)
    ax.view_init(elev=22, azim=-58)
    set_axes_equal(ax); ax.set_axis_off()
plt.show()

The recovered viewpoints agree with what the photographs suggest: one is higher and farther back, while the other is lower and turned towards the gable.

## Algebraic and geometric error

The DLT estimates the camera by solving

$$
\min_{\|\mathbf p\|=1}\|\mathsf A\mathbf p\|^2.
$$

This is an **algebraic error**: it measures how well the entries of
$\mathbf p$ satisfy the homogeneous linear equations. Its main advantage is
precisely that it leads to a linear solution.

What we ultimately care about, however, is what happens in the image. Let

$$
\pi(X,Y,Z)^\top
=
\left(\frac{X}{Z},\frac{Y}{Z}\right)^\top
$$

denote conversion from homogeneous to ordinary image coordinates. The
reprojection error of one correspondence is

$$
\left\|
\mathbf x_i -
\pi(\mathsf P\mathbf X_i)
\right\|,
$$

and a natural refinement of the camera is therefore

$$
\min_{\mathsf P}
\sum_{i=1}^n
\left\|
\mathbf x_i -
\pi(\mathsf P\mathbf X_i)
\right\|^2.
$$

Here the observed image points $\mathbf x_i$ and the 3D points
$\mathbf X_i$ are held fixed; only the camera is varied. The problem is no
longer linear because of the division by the third coordinate in $\pi$.

Unlike the algebraic residual, the reprojection error has a direct unit and
interpretation: pixels. Under the usual model in which the 3D points are exact
and the image measurements are affected by isotropic Gaussian noise, minimizing
squared reprojection error is also the natural statistical objective.

There is no closed-form solution in general. A common strategy is therefore to
use the DLT estimate as an initial camera and refine it with a nonlinear
least-squares method.

A small technicality remains. Reprojection is unchanged if we replace
$\mathsf P$ by $\alpha\mathsf P$, so the nonlinear problem still has the same
projective scale ambiguity.

To remove it, we scale the initial camera so that one of its entries is equal
to one, keep that entry fixed, and optimize the remaining eleven parameters.
The DLT estimate provides the starting point.

In [ ]:
from scipy.optimize import least_squares


def refine_camera(P0, X, x):
    """Refine a projective camera by minimising reprojection error."""
    P0 = np.asarray(P0, float)

    # Fix the projective scale using the largest entry of the DLT estimate.
    fixed = np.argmax(np.abs(P0))
    P0 = P0 / P0.flat[fixed]

    free = np.ones(12, dtype=bool)
    free[fixed] = False

    def unpack(theta):
        p = np.empty(12)
        p[fixed] = 1.0
        p[free] = theta
        return p.reshape(3, 4)

    def residuals(theta):
        P = unpack(theta)
        return (project_points(X, P) - x).ravel()

    result = least_squares(
        residuals,
        P0.ravel()[free],
    )

    return unpack(result.x)

In [ ]:
P1_ref = refine_camera(P1, X, x1)
P2_ref = refine_camera(P2, X, x2)

for (name, _, x), P_dlt, P_ref in zip(
    views, [P1, P2], [P1_ref, P2_ref]
):
    before = rms(reprojection_error(P_dlt, X, x))
    after  = rms(reprojection_error(P_ref, X, x))

    print(f"{name}: {before:.2f} px  ->  {after:.2f} px")

While the DLT found a
projective camera by minimising an algebraic residual, nonlinear least squares instead
starts from that estimate and moves the same eleven projective degrees of
freedom to reduce reprojection error directly.

On the real house, a smaller reprojection error should not be interpreted as
recovering a physically exact camera: the image annotations are noisy and the
3D model itself is only nominal.

## When the points are coplanar

There was one qualification in the six-point count: the 3D points must be in
general position. The most important failure case is when they all lie on a
plane.

Take, for example, the front wall of the house, $Y=0$. A point on that plane has
coordinates

$$
\mathbf X=(X,0,Z,1)^\top.
$$

Writing the camera by columns,

$$
\mathsf P=
\begin{bmatrix}
\mathbf p_1 & \mathbf p_2 & \mathbf p_3 & \mathbf p_4
\end{bmatrix},
$$

its projection becomes

$$
\mathsf P\mathbf X
=
\begin{bmatrix}
\mathbf p_1 & \mathbf p_3 & \mathbf p_4
\end{bmatrix}
\begin{bmatrix}
X\\
Z\\
1
\end{bmatrix}.
$$

The column $\mathbf p_2$ has disappeared completely. No number of observations
on this plane can tell us how the camera acts in the direction away from it.

What the correspondences determine instead is the plane-to-image homography

$$
\mathsf H=
\begin{bmatrix}
\mathbf p_1 & \mathbf p_3 & \mathbf p_4
\end{bmatrix}.
$$

Thus planar correspondences can determine how that particular plane appears in
the image, but they cannot determine a unique projective camera in 3D.

For sufficiently many generic points on the plane, the homography has eight
degrees of freedom. The remaining camera column contributes three completely
unconstrained parameters. Accordingly, the $12$ camera coefficients are
constrained only up to a four-dimensional null space, and the design matrix has
rank at most $8$.

There is one small trap when trying to demonstrate this numerically. With only
four points, any $8\times12$ design matrix has a null space of dimension at
least four simply by counting, whether the points are planar or not. To expose
the degeneracy we must use enough correspondences that a non-coplanar
configuration would determine the camera, at least six.

In [ ]:
rng = np.random.default_rng(0)


def sample_on_wall(n):
    """Points on the front wall Y = 0 of the house, as columns."""
    lo, hi = X_ALL.min(axis=1), X_ALL.max(axis=1)
    return np.vstack((
        rng.uniform(lo[0], hi[0], n),
        np.zeros(n),
        rng.uniform(lo[2], hi[2], n),
    ))


def sample_in_space(n):
    """Points inside the bounding box of the house, as columns."""
    lo, hi = X_ALL.min(axis=1), X_ALL.max(axis=1)
    return rng.uniform(lo, hi, size=(n, 3)).T


def camera_rank(X):
    x = project_points(X, P_syn)
    Xn, _ = normalise_3d(X)
    xn, _ = normalise_2d(x)
    return np.linalg.matrix_rank(camera_design_matrix(Xn, xn), tol=1e-9)


print(f"{'points':>8s}{'planar rank':>15s}{'spatial rank':>15s}")

for n in [4, 6, 8, 12, 30]:
    rp = camera_rank(sample_on_wall(n))
    rs = camera_rank(sample_in_space(n))

    print(f"{n:8d}{rp:15d}{rs:15d}")

Once enough points are available, the difference is unambiguous. Points spread
through space give the expected rank $11$, while points confined to one plane
stop at rank $8$: three additional camera parameters remain completely
unobservable.

In [ ]:
#| echo: false
#| column: page
#| label: fig-coplanar
#| fig-cap: >-
#|   A planar configuration cannot determine a general projective camera.
#|   **Left:** among the annotated house corners, four lie on the front wall
#|   while the others provide information away from that plane.
#|   **Right:** singular values of the normalized DLT system for twelve points
#|   spread through space and twelve points confined to a plane. The generic
#|   3D configuration has one null direction, corresponding to the global
#|   scale of the camera; the planar configuration has four.

def normalised_design_matrix(X, x):
    Xn, _ = normalise_3d(X)
    xn, _ = normalise_2d(x)
    return camera_design_matrix(Xn, xn)


def spectrum(A):
    """Singular values relative to the largest."""
    s = np.linalg.svd(A, compute_uv=False)
    return s / s[0]


# Two exact synthetic configurations seen by the same camera.
X_space = sample_in_space(12)
X_wall  = sample_on_wall(12)

A_space = normalised_design_matrix(X_space, project_points(X_space, P_syn))
A_wall  = normalised_design_matrix(X_wall,  project_points(X_wall, P_syn))


fig = plt.figure(figsize=(13.0, 4.8), layout="constrained")

# --- Geometry ---------------------------------------------------------------

ax = fig.add_subplot(1, 2, 1, projection="3d")

wireframe_3d(ax, X_ALL, color=GREY, lw=1.2)

front = [i for i, k in enumerate(ids) if np.isclose(V3[k][1, 0], 0.0)]
off_front = [i for i in range(X.shape[1]) if i not in front]

# Order the coplanar points around their centroid so the outline closes.
Q = X[:, front]
order = np.argsort(np.arctan2(Q[2] - Q[2].mean(), Q[0] - Q[0].mean()))
quad = Q[:, order]
ax.plot(*np.hstack([quad, quad[:, :1]]), color=BLUE, lw=2.0)

ax.scatter(
    *X[:, front],
    s=80,
    color=BLUE,
    label="on the front wall",
)

ax.scatter(
    *X[:, off_front],
    s=80,
    color=ACCENT,
    label="off the plane",
)

ax.legend(fontsize=9, frameon=False, loc="upper left")
ax.view_init(elev=18, azim=-64)
set_axes_equal(ax)
ax.set_axis_off()


# --- Singular values --------------------------------------------------------

ax = fig.add_subplot(1, 2, 2)

k = np.arange(1, 13)

ax.semilogy(
    k,
    np.maximum(spectrum(A_space), 1e-16),
    "o-",
    lw=1.8,
    color=DARK,
    label="twelve points in space",
)

ax.semilogy(
    k,
    np.maximum(spectrum(A_wall), 1e-16),
    "s--",
    lw=1.8,
    color=BLUE,
    label="twelve points on a plane",
)

ax.set_xticks(k)
ax.set_xlabel("index of the singular value")
ax.set_ylabel("singular value, relative to the largest")
ax.legend(fontsize=9, frameon=False)
ax.spines[["top", "right"]].set_visible(False)

plt.show()

The difference is also visible in the singular values of the design matrix.
With exact generic 3D points, only one singular value vanishes: the camera is
determined up to its global scale. With planar points, four singular values
vanish, leaving a four-dimensional null space.

The extra three null directions are exactly the three coefficients of the
camera column that never appears when all points lie on the plane. What the
data determine is only the plane-to-image homography.

## Beyond coplanarity: the twisted cubic

Coplanarity is the most familiar degeneracy in camera resection, but it is not
the only one.

A classical critical configuration occurs when the 3D points and the camera
centre lie on the same **twisted cubic**. In projective coordinates, such a
curve can be written as

$$
\Gamma(t)
\sim
\mathsf A
\begin{bmatrix}
1 & t & t^2 & t^3
\end{bmatrix}^{\mathsf T},
$$

for some invertible $4\times4$ matrix $\mathsf A$.

For points on such a configuration, the image correspondences do not determine
a unique projective camera. Instead of the usual one-dimensional null space
corresponding only to global scale, the DLT system has an additional null
direction. Projectively, this gives a one-parameter family of cameras that
reproject the same data exactly.

There are also reducible critical cubics. One simple case is already visible
from the planar degeneracy discussed above. Points confined to a plane can
constrain the design matrix only up to rank eight. If five generic points lie
on that plane, adding a sixth point outside it contributes at most two further
independent equations:

$$
8+2=10.
$$

The camera therefore remains underdetermined: a minimal six-point set with five
coplanar points is degenerate even though the six points together span
$\mathbb P^3$.

This is a useful warning about the phrase *general position*. Merely checking
that the 3D points are not all coplanar is not enough to guarantee a valid
minimal resection configuration.

In [ ]:
#| echo: false
#| column: page
#| label: fig-critical-configurations
#| fig-cap: >-
#|   Three critical configurations for projective camera resection. Grey points
#|   show the centres of different cameras that reproject the data exactly.
#|   **Left:** when all control points lie on a plane, the design matrix has
#|   nullity four; after accounting for global scale, three degrees of camera
#|   ambiguity remain, and the possible centres fill a three-dimensional set.
#|   **Centre:** when the points and the camera centre lie on a twisted cubic,
#|   the nullity is two and the resulting one-parameter family of camera centres
#|   lies on the same cubic. **Right:** with five coplanar points and one point
#|   off the plane, the nullity is again two; the ambiguous centres lie on the
#|   line through the original camera centre and the off-plane point.

# A camera closer to the scene, used for this figure only.
house_centre = X_ALL.mean(axis=1, keepdims=True)
C_fig = house_centre + np.array([[6.0], [-9.0], [6.0]])
R_fig = look_at(C_fig, house_centre)
P_fig = intrinsic_matrix(1400.0, cx=W_PIX / 2, cy=H_PIX / 2) @ np.hstack(
    (R_fig, -R_fig @ C_fig)
)


def cubic_through(points, ts):
    """The twisted cubic through four points, at four chosen parameter values."""
    V = np.vstack([np.ones(4), ts, ts**2, ts**3])
    return np.vstack((points, np.ones((1, 4)))) @ np.linalg.inv(V)


def on_cubic(A, ts):
    """Points on the cubic, as columns."""
    Q = A @ np.vstack([np.ones_like(ts), ts, ts**2, ts**3])
    return Q[:3] / Q[3]


rng = np.random.default_rng(0)

X_plane = np.vstack((rng.uniform(0, 5, 8), np.zeros(8), rng.uniform(0, 2.5, 8)))

control = np.hstack([
    C_fig,
    np.array([[0.0], [4.5], [3.0]]),
    np.array([[3.0], [0.5], [-1.0]]),
    np.array([[6.0], [-3.0], [4.5]]),
])
A_cubic = cubic_through(control, np.array([0.0, 1.0, 2.0, 3.0]))
X_cubic = on_cubic(A_cubic, np.linspace(0.5, 3.0, 8))

X_off = np.array([[3.0], [3.6], [2.0]])
X_five = np.hstack([
    np.vstack((rng.uniform(0, 5, 5), np.zeros(5), rng.uniform(0, 2.5, 5))),
    X_off,
])


def near(C, radius=9.0):
    return np.all(np.abs(C - C_fig) < radius)


fig = plt.figure(figsize=(13.5, 4.6), layout="constrained")

wall_x, wall_z = np.meshgrid([0.0, 5.0], [-0.3, 2.8])
wall_y = np.zeros_like(wall_x)

# --- points on a plane: the free column gives a three-parameter family -------
ax = fig.add_subplot(1, 3, 1, projection="3d")
ax.plot_surface(wall_x, wall_y, wall_z, color=BLUE, alpha=0.13)
ax.scatter(*X_plane, s=40, color=BLUE, depthshade=False, zorder=6)

x_plane = project_points(X_plane, P_fig)
cloud = []
for _ in range(9000):
    P = P_fig + np.outer(rng.normal(0, 120, 3), [0, 1, 0, 0])
    C = camera_centre(P)
    if near(C) and reprojection_error(P, X_plane, x_plane).max() < 1e-6:
        cloud.append(C[:, 0])
ax.scatter(*np.array(cloud).T, s=4, color=GREY, alpha=0.30, depthshade=False)

draw_camera(ax, P_fig, size=2.4)
ax.set_title("points on a plane\nfour-dimensional null space", fontsize=10)

# --- points and centre on a twisted cubic -----------------------------------
ax = fig.add_subplot(1, 3, 2, projection="3d")
ax.plot(*on_cubic(A_cubic, np.linspace(-0.15, 3.3, 400)), color=BLUE, lw=1.5,
        alpha=0.85)
ax.scatter(*X_cubic, s=40, color=BLUE, depthshade=False, zorder=6)

x_cubic = project_points(X_cubic, P_fig)
N = np.linalg.svd(camera_design_matrix(X_cubic, x_cubic))[2][-2:]
centres = []
for a in np.linspace(0, np.pi, 1500):
    P = (np.cos(a) * N[0] + np.sin(a) * N[1]).reshape(3, 4)
    C = camera_centre(P)
    if near(C) and reprojection_error(P, X_cubic, x_cubic).max() < 1e-4:
        centres.append(C[:, 0])
ax.scatter(*np.array(centres).T, s=5, color=GREY, alpha=0.8, depthshade=False)

draw_camera(ax, P_fig, size=2.4)
ax.set_title("points and centre on a twisted cubic\ntwo-dimensional null space", fontsize=10)

# --- five on a plane, one off it --------------------------------------------
ax = fig.add_subplot(1, 3, 3, projection="3d")
ax.plot_surface(wall_x, wall_y, wall_z, color=BLUE, alpha=0.13)
ax.scatter(*X_five[:, :5], s=40, color=BLUE, depthshade=False, zorder=6)
ax.scatter(*X_off, s=55, color=BLUE, depthshade=False, zorder=6)

direction = X_off - C_fig
ax.plot(*np.hstack([C_fig - 0.25 * direction, C_fig + 2.2 * direction]),
        color=BLUE, lw=1.3, ls=":", alpha=0.9)

x_five = project_points(X_five, P_fig)
q6 = (P_fig @ np.vstack((X_off, [[1.0]])))[:, 0]
D = np.zeros((3, 4))
D[:, 1] = q6                      # only the column that the plane never sees

centres = []
for a in np.linspace(-2, 2, 3000):
    P = P_fig + a * D
    C = camera_centre(P)
    if near(C) and reprojection_error(P, X_five, x_five).max() < 1e-6:
        centres.append(C[:, 0])
ax.scatter(*np.array(centres).T, s=5, color=GREY, alpha=0.8, depthshade=False)

draw_camera(ax, P_fig, size=2.4)
ax.set_title("five on a plane, one off it\ntwo-dimensional null space", fontsize=10)

for ax in fig.axes:
    ax.view_init(elev=18, azim=-64)
    ax.set_xlim(-3, 10); ax.set_ylim(-10, 6); ax.set_zlim(-2, 9)
    ax.set_box_aspect((13, 16, 11), zoom=1.5)
    ax.set_axis_off()

plt.show()

The three cases differ in how much freedom remains after the image
correspondences have been fixed.

For points on the plane $Y=0$, the column $\mathbf p_2$ of the camera never
appears in the projection equations. It can therefore be changed by an
arbitrary three-vector without affecting any image point. This gives the
four-dimensional null space seen above: one dimension is the usual global
scale of $\mathsf P$, while the other three correspond to genuine camera
ambiguity.

Adding one point $\mathbf X_6$ away from the plane removes two of those three
degrees of freedom. One ambiguity remains. If

$$
\mathbf q_6=\mathsf P\mathbf X_6,
$$

then the family

$$
\mathsf P(\alpha)
=
\mathsf P
+
\alpha\,\mathbf q_6\,\mathbf e_2^\top
$$

produces exactly the same image measurements. For every planar point,
$\mathbf e_2^\top\mathbf X=Y=0$, so its projection is unchanged. For the
sixth point,

$$
\mathsf P(\alpha)\mathbf X_6
=
\bigl(1+\alpha Y_6\bigr)\,
\mathsf P\mathbf X_6,
$$

which differs only by homogeneous scale.

The corresponding camera centres lie on the projective line through the
original centre $\mathbf C$ and the off-plane point $\mathbf X_6$. Thus the
remaining one-parameter ambiguity has a simple geometric form: the viewpoint
can move along that line without changing any of the six image measurements.

The twisted-cubic case has the same nullity, but a different geometry. The
one-parameter family of ambiguous camera centres is no longer a line: it lies
on the twisted cubic containing the control points and the original camera
centre.

In all three cases the reprojection error is exactly zero. The ambiguity is
therefore not caused by noise or by the estimation algorithm: the image data
themselves do not determine a unique camera. 

## Questions to leave open

**Five and a half.** The counting suggests that five complete correspondences
plus one additional scalar constraint should be enough. Build the system using
five image points and only the $u$ coordinate of a sixth, and check that its
null space is generically one-dimensional.

What does that last constraint mean geometrically? Can you rewrite it as saying
that the projection of the sixth 3D point must lie on a known image line?

**Which six?** With ten vertices available there are $\binom{10}{6}=210$
minimal subsets. Compute the rank of the normalized design matrix for each one.
Are all six-point subsets usable?

Among the non-degenerate subsets, use the second-smallest singular value
$\sigma_{11}$ as a measure of distance from degeneracy. Does it predict which
subsets are most sensitive to a few pixels of image noise?

**Near a critical configuration.** On an exact twisted cubic the DLT has a
two-dimensional null space. Perturb the 3D points slightly away from the curve
and recompute the normalized design matrix.

How does the second-smallest singular value $\sigma_{11}$ change as the
perturbation grows? What happens to the sensitivity of the recovered camera
centre to small image noise?

**Whose error is it?** On the real photographs, reprojection error mixes two
effects: uncertainty in the annotated image points and mismatch between the
ideal 3D model and the folded paper house.

Can these two sources of error be separated from the images alone? What
additional assumptions or measurements would make the problem identifiable?

As a more constrained experiment, allow only a small number of physically
meaningful deformations of the house — for example its fold angles — and test
whether the resulting model improves reprojection on views that were not used
for fitting.

## Further reading

- Fusiello, A. Computer Vision: *Three-dimensional Reconstruction Techniques*, Springer Cham, 2024.
- Hartley, R. and Zisserman, A. *Multiple View Geometry in Computer Vision*, 2nd ed., Cambridge University Press, 2004. See Chapter 7 for camera
  resection and normalization, Chapter 22 for degenerate configurations,
  and Chapter 6 for camera models and the decomposition of the camera matrix.
- Abdel-Aziz, Y. I. and Karara, H. M. "Direct linear transformation from comparator coordinates into object space coordinates in close-range photogrammetry", *Proceedings of the ASP Symposium on Close-Range Photogrammetry*, 1971. Where the DLT comes from, and it is photogrammetry rather than computer vision.

---

**Luca Magri** — Computer Vision Dojo  
Code MIT · text and figures CC BY-NC-ND 4.0  
<https://magrilu.github.io/cv-dojo/>